In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "310_build_gold_fact_project_item_bid.py"
RUN_ID = str(uuid.uuid4())

FACT_BID_ITEM_TABLE = f"{catalog}.silver.fact_bid_item"
FACT_BID_PROJECT_TABLE = f"{catalog}.silver.fact_bid_project"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
DIM_ITEM_SPEC_TABLE = f"{catalog}.silver.dim_item_specification"
DIM_ITEM_WORK_CATEGORY_TABLE = f"{catalog}.gold.dim_item_work_category"
TARGET_TABLE = f"{catalog}.gold.fact_project_item_bid"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_item_bid
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH work_category_map AS (
        SELECT
              effective_specification_description_standardized
            , item_work_category
            , item_work_category_source
            , is_defaulted_work_category
        FROM (
            SELECT
                  effective_specification_description_standardized
                , item_work_category
                , item_work_category_source
                , is_defaulted_work_category
                , ROW_NUMBER() OVER (
                    PARTITION BY effective_specification_description_standardized
                    ORDER BY
                          is_defaulted_work_category ASC
                        , item_work_category_source
                        , specification_description
                        , effective_specification_description_standardized
                  ) AS rn
            FROM {DIM_ITEM_WORK_CATEGORY_TABLE}
        ) x
        WHERE rn = 1
    )

    , project_bid_unique AS (
        SELECT *
        FROM (
            SELECT
                  fp.*
                , ROW_NUMBER() OVER (
                    PARTITION BY fp.project_id, fp.vendor_name_standardized
                    ORDER BY
                          fp.project_bid_rank ASC NULLS LAST
                        , fp.max_bid_total_amount ASC NULLS LAST
                        , fp.bid_project_fact_key DESC NULLS LAST
                  ) AS rn
            FROM {FACT_BID_PROJECT_TABLE} fp
        ) x
        WHERE rn = 1
    )

    , src AS (
        SELECT
              f.bid_item_fact_key
            , f.project_key
            , f.vendor_key
            , f.item_specification_key

            , f.project_id
            , f.vendor_name
            , f.vendor_name_standardized

            , COALESCE(
                  f.project_classification_key
                , fp.project_classification_key
              ) AS project_classification_key

            , COALESCE(
                  f.county_key
                , fp.county_key
              ) AS county_key

            , dp.project_name
            , dp.project_number
            , dp.state_project_number
            , dp.control_section_job_csj
            , dp.controlling_project_id_ccsj
            , dp.project_type
            , dp.project_classification
            , dp.project_actual_let_date
            , dp.project_estimated_let_date
            , dp.project_limits_from
            , dp.project_limits_to
            , dp.county
            , dc.county_location AS location
            , dp.district_division
            , dp.highway
            , dp.short_description
            , dp.federal_project_number

            , f.bid_submit_sequence_number
            , f.bid_rank_sequence_number
            , f.low_bidder_flag
            , f.low_bidder_flag_int
            , f.dbe_goal_percent
            , f.maximum_number_of_working

            , f.bid_code
            , f.alternative_bid_code
            , f.bid_item_sequence_number
            , f.bid_item_description
            , f.measurement_unit
            , f.bid_item_quantity
            , f.bid_item_unit_price_amount
            , f.bid_total_amount

            , f.specification_code
            , f.specification_description
            , ds.effective_specification_description
            , ds.effective_specification_description_standardized
            , f.spec_book_year

            , wc.item_work_category
            , wc.item_work_category_source
            , wc.is_defaulted_work_category

            , f.force_account_amount
            , f.force_account_description
            , f.nbi_number
            , f.utility_id

            , f.business_submission_key
            , f.business_item_key

            , f.item_version_count
            , f.min_submit_sequence_for_item
            , f.max_submit_sequence_for_item
            , f.vendor_project_submit_sequence_count
            , f.latest_submit_for_vendor_project

            , f.vendor_project_has_resubmission_history
            , f.item_has_resubmission_history
            , f.item_changed_across_submit_sequences
            , f.is_latest_submit_for_vendor_project
            , f.item_missing_from_latest_project_submit
            , f.item_changed_value_across_submits

            , f.is_standard_specification_item
            , f.is_unmapped_specification_item
            , f.is_payment_adjustment_item
            , f.is_special_deduction_item
            , f.is_nonstandard_adjustment_item

            , f.extended_item_amount_calc
            , f.project_total_vs_extended_item_abs_diff

            , fp.is_low_bidder AS is_low_bid_project_vendor
            , fp.project_bid_rank
            , fp.bidder_count_in_project
            , fp.lowest_bid_amount_in_project
            , fp.highest_bid_amount_in_project
            , fp.bid_spread_amount_in_project
            , fp.bid_vs_low_bid_ratio
            , fp.max_bid_total_amount AS vendor_project_bid_total
            , fp.min_bid_total_amount AS vendor_project_bid_total_min
            , fp.has_conflicting_bid_total_amounts

            , CASE
                  WHEN fp.lowest_bid_amount_in_project IS NOT NULL
                   AND fp.lowest_bid_amount_in_project <> 0
                   AND f.extended_item_amount_calc IS NOT NULL
                  THEN f.extended_item_amount_calc / fp.lowest_bid_amount_in_project
                  ELSE NULL
              END AS item_amount_as_pct_of_low_bid_project_total

            , CASE
                  WHEN fp.max_bid_total_amount IS NOT NULL
                   AND fp.max_bid_total_amount <> 0
                   AND f.extended_item_amount_calc IS NOT NULL
                  THEN f.extended_item_amount_calc / fp.max_bid_total_amount
                  ELSE NULL
              END AS item_amount_as_pct_of_vendor_bid_total

            , f.source_name
            , f.source_row_id
            , f.source_created_at
            , f.source_updated_at
            , f.source_version
            , f.ingestion_run_id
            , f.ingested_at_utc
            , f.record_hash

        FROM {FACT_BID_ITEM_TABLE} f
        LEFT JOIN {DIM_PROJECT_TABLE} dp
            ON f.project_id = dp.project_id
        LEFT JOIN {DIM_COUNTY_TABLE} dc
            ON dp.county = dc.county
        LEFT JOIN {DIM_ITEM_SPEC_TABLE} ds
            ON f.item_specification_key = ds.item_specification_key
        LEFT JOIN work_category_map wc
            ON ds.effective_specification_description_standardized = wc.effective_specification_description_standardized
        LEFT JOIN project_bid_unique fp
            ON f.project_id = fp.project_id
           AND f.vendor_name_standardized = fp.vendor_name_standardized
    )

    SELECT *
    FROM src
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built gold.fact_project_item_bid successfully."
    )

    print("=" * 70)
    print("Built gold.fact_project_item_bid")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise